# EDA датасета AUTO.RIA Numberplate (UA)

**Связь с ДЗ12:** повторяем подход анализа датасета, как для GTSRB — распределения, балансировка, визуализация.

**Вопросы, на которые отвечаем:**
1. Сколько изображений и аннотаций в train / val?
2. Сколько номеров на картинке (одиночные vs групповые)?
3. Каков aspect ratio и относительный размер номера?
4. Какова яркость изображений (равномерно ли представлены разные условия освещения)?
5. Примеры из выборки.

In [ ]:
from pathlib import Path
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import random, yaml

sns.set_theme(style='whitegrid')
CFG = yaml.safe_load(open('../config.yaml', encoding='utf-8'))
DATA_DIR = Path(CFG['yolo_dataset_dir'])
print('Dataset root:', DATA_DIR)

In [ ]:
def collect_split(split: str):
    imgs = sorted(list((DATA_DIR / 'images' / split).glob('*.jpg')) + list((DATA_DIR / 'images' / split).glob('*.png')))
    rows = []
    for img_path in imgs:
        img = cv2.imread(str(img_path))
        if img is None: continue
        H, W = img.shape[:2]
        brightness = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean()
        lbl = DATA_DIR / 'labels' / split / f'{img_path.stem}.txt'
        boxes = lbl.read_text().strip().splitlines() if lbl.exists() else []
        for b in boxes:
            _, cx, cy, bw, bh = map(float, b.split())
            rows.append(dict(split=split, img=img_path.name, W=W, H=H,
                              bw_px=bw*W, bh_px=bh*H,
                              aspect=(bw*W)/max(bh*H,1),
                              rel_area=bw*bh,
                              brightness=brightness,
                              n_plates_in_img=len(boxes)))
    return pd.DataFrame(rows)

df_train = collect_split('train')
df_val = collect_split('val')
print(f'train: {df_train.img.nunique()} imgs, {len(df_train)} plates')
print(f'val:   {df_val.img.nunique()} imgs, {len(df_val)} plates')

## 1. Плотность — сколько номеров на одно фото

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, df, name in zip(ax, [df_train, df_val], ['train', 'val']):
    counts = df.groupby('img')['n_plates_in_img'].first()
    counts.value_counts().sort_index().plot.bar(ax=a)
    a.set_title(f'{name}: номеров на картинку')
    a.set_xlabel('кол-во номеров'); a.set_ylabel('фото')
plt.tight_layout(); plt.show()

## 2. Aspect ratio и размер номера

Украинский стандартный номер имеет соотношение сторон ~3.2:1 для 2-рядного (мото) и ~4.6:1 для авто-однорядного. Посмотрим, что показывает датасет.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df_train.aspect.hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Aspect ratio (W/H) плит'); axes[0].set_xlabel('ratio')
df_train.rel_area.hist(bins=50, ax=axes[1], color='coral')
axes[1].set_title('Relative area plate / image'); axes[1].set_xlabel('frac')
axes[2].scatter(df_train.bw_px, df_train.bh_px, s=3, alpha=0.3)
axes[2].set_title('bbox px: W vs H'); axes[2].set_xlabel('W px'); axes[2].set_ylabel('H px')
plt.tight_layout(); plt.show()

print('Aspect median =', df_train.aspect.median().round(2))
print('Relative area median =', df_train.rel_area.median().round(4))

## 3. Яркость изображений

Если распределение смещено (ночных почти нет) — это повод усилить аугментацию HSV в тренировке.

In [ ]:
plt.figure(figsize=(10, 3))
df_train.drop_duplicates('img').brightness.hist(bins=40, color='mediumseagreen')
plt.title('Средняя яркость по изображениям (train)')
plt.xlabel('mean grayscale'); plt.ylabel('фото')
plt.axvline(df_train.brightness.mean(), color='red', linestyle='--', label=f'mean={df_train.brightness.mean():.0f}')
plt.legend(); plt.show()

## 4. Примеры из выборки (с отрисовкой bbox)

In [ ]:
def draw(img_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    lbl = DATA_DIR / 'labels/train' / f'{img_path.stem}.txt'
    for line in lbl.read_text().splitlines():
        _, cx, cy, bw, bh = map(float, line.split())
        x1 = int((cx - bw/2) * W); y1 = int((cy - bh/2) * H)
        x2 = int((cx + bw/2) * W); y2 = int((cy + bh/2) * H)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    return img

samples = random.sample(list((DATA_DIR / 'images/train').glob('*.jpg')), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, p in zip(axes.ravel(), samples):
    ax.imshow(draw(p)); ax.axis('off'); ax.set_title(p.name[:20])
plt.tight_layout(); plt.show()

## Выводы для обучения

- Подавляющее большинство — одиночные номера на фото.
- Aspect ratio сконцентрирован в районе 3–5 (стандартная форма).
- Яркость относительно равномерная, но стоит усилить HSV-аугментацию (hsv_v=0.3 в 02_train_detector.ipynb).
- `fliplr=0.0` — зеркальное отражение ломает текст номера, отключено.
